<a href="https://colab.research.google.com/github/SophiArch/Briefings/blob/main/Agent/agent_review_lab/reference_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reviewing the agent's work — instructor answer key

`agent_v1_output.ipynb` claims **98.9% accuracy, ready for production**. Every step in it ran without error,
and every number is real. It is still wrong.

This notebook walks through the review, one checklist item at a time, and ends with the numbers we would actually report
against the fraud team's business goal: *"flag at least 80% of confirmed fraud by value"*.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

raw = pd.read_csv("https://raw.githubusercontent.com/JasonL888/AI_Experiments/refs/heads/main/Datasets/fraud_dataset.csv")

def build_features(d):
    t = pd.to_datetime(d.trans_date_trans_time, format="%d-%m-%Y %H:%M")
    X = pd.DataFrame({
        "amt": d.amt,
        "hour": t.dt.hour,
        "age": (t - pd.to_datetime(d.dob, format="%d-%m-%Y")).dt.days // 365,
        "city_pop": d.city_pop,
        "lat": d.lat, "long": d.long,
        "merch_lat": d.merch_lat, "merch_long": d.merch_long,
        "male": (d.gender == "M").astype(int),
    })
    return pd.concat([X, pd.get_dummies(d.category, prefix="cat", dtype=int)], axis=1)

## Check 1 — Reproduce the claim

Before arguing with a number, make sure you get the same one.

In [2]:
def evaluate(d, train_idx, test_idx, label):
    X, y = build_features(d), d.is_fraud.astype(int)
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X.iloc[train_idx], y.iloc[train_idx])
    p, yt = m.predict(X.iloc[test_idx]), y.iloc[test_idx].values
    amt = d.amt.iloc[test_idx].values
    return {
        "setup": label,
        "test rows": len(test_idx),
        "accuracy": accuracy_score(yt, p),
        "majority baseline": max(yt.mean(), 1 - yt.mean()),
        "recall (count)": recall_score(yt, p),
        "precision": precision_score(yt, p),
        "recall (by $ value)": amt[(p == 1) & (yt == 1)].sum() / amt[yt == 1].sum(),
    }

def random_split(d):
    return train_test_split(np.arange(len(d)), test_size=0.3, stratify=d.is_fraud, random_state=42)

results = [evaluate(raw, *random_split(raw), "1. agent: random split, as given")]
pd.DataFrame(results).round(3)

,setup,test rows,accuracy,majority baseline,recall (count),precision,recall (by $ value)
0,"1. agent: random split, as given",2479,0.989,0.5,0.995,0.983,1.0


Reproduced: 98.9% accuracy, 99.5% recall. Now: **do we believe it?**

## Check 2 — Duplicates (the agent dropped `trans_num` before looking at it)

`trans_num` is a transaction ID. The agent called it "no predictive signal" and dropped it, which also destroyed the
only way to see that rows are repeated.

In [3]:
print(f"rows                 {len(raw):,}")
print(f"distinct trans_num   {raw.trans_num.nunique():,}")
print(f"exact duplicate rows {raw.duplicated().sum():,}")
print("label of duplicate rows:", raw[raw.duplicated(subset='trans_num')].is_fraud.value_counts().to_dict())

rows                 8,262
distinct trans_num   5,415
exact duplicate rows 2,847
label of duplicate rows: {1.0: 2847}


In [4]:
# How many test frauds have an identical twin in the training set?
tr, te = random_split(raw)
train_ids = set(raw.trans_num.iloc[tr])
test_fraud = raw.iloc[te].query("is_fraud == 1")
print(f"test fraud rows with a twin in train: {test_fraud.trans_num.isin(train_ids).mean():.0%}")

test fraud rows with a twin in train: 86%


Every duplicate is a fraud. The "perfect balance" the agent celebrated was **manufactured by copying fraud rows**,
and most test frauds were memorised, not predicted. A classic trap.

## Check 3 — Baseline

After removing duplicates, what does a model that always says "genuine" score?

In [5]:
dedup = raw.drop_duplicates(subset="trans_num").reset_index(drop=True)
print(dedup.is_fraud.value_counts().to_dict())
print(f"majority-class baseline accuracy: {1 - dedup.is_fraud.mean():.1%}")
results.append(evaluate(dedup, *random_split(dedup), "2. deduplicated, random split"))

{0.0: 4131, 1.0: 1284}
majority-class baseline accuracy: 76.3%


The honest baseline is **76%**, not 50%. Accuracy has to be judged against that, which is why it is the wrong headline.

## Check 4 — Same card on both sides of the split

Fraud comes in **bursts on the same card**. With a random split, the model can learn a stolen card's pattern from
some of its transactions and "predict" the rest. In production, a new stolen card has no history.

In [6]:
print(f"cards: {dedup.cc_num.nunique()}, fraud transactions per compromised card: "
      f"{dedup.query('is_fraud == 1').groupby('cc_num').size().median():.0f} (median)")

tr, te = random_split(dedup)
fraud_cards_test = set(dedup.iloc[te].query("is_fraud == 1").cc_num)
fraud_cards_train = set(dedup.iloc[tr].query("is_fraud == 1").cc_num)
print(f"compromised cards in test that also appear as fraud in train: {len(fraud_cards_test & fraud_cards_train) / len(fraud_cards_test):.0%}")

cards: 306, fraud transactions per compromised card: 10 (median)
compromised cards in test that also appear as fraud in train: 99%


In [7]:
# Group split: every card goes entirely to train or entirely to test
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(dedup, groups=dedup.cc_num))
results.append(evaluate(dedup, tr, te, "3. deduplicated, split by card"))

## Check 5 — Does the metric match the business goal?

The goal is **at least 80% of confirmed fraud by value**, not accuracy. Precision matters too: every false alert is a
case a Fraud Analyst has to read.

In [8]:
summary = pd.DataFrame(results).set_index("setup")
summary.style.format({c: "{:.1%}" for c in summary.columns if c != "test rows"})

,test rows,accuracy,majority baseline,recall (count),precision,recall (by $ value)
setup,,,,,,
"1. agent: random split, as given",2479,98.9%,50.0%,99.5%,98.3%,100.0%
"2. deduplicated, random split",1625,97.1%,76.3%,89.6%,98.0%,98.4%
"3. deduplicated, split by card",1910,96.2%,78.4%,84.5%,97.5%,98.9%


## What we would actually report

| | Agent | Reviewed |
|---|---|---|
| Headline | "99% accurate, ready for production" | Catches **98.9% of fraud by value**, which beats the 80% goal |
| Fraud caught (count) | 99.5% | **84.5%** on cards the model has never seen. Small-value frauds are the ones missed |
| Baseline | not reported | 78% of test rows are genuine (the "always genuine" score) |
| Caveats | none | Source file oversampled fraud by duplication; synthetic data; needs a time-based test before production |

The model is **useful**, and the honest result still meets the business goal. But the agent's number was inflated, its
evidence was wrong, and "ready for production" was a claim it had no basis to make.

**Lesson:** the agent was fast and its code ran. The judgement — what to check, which split mirrors production, which
metric the business cares about — came from the reviewer.